# Sentinel on REAL data — Colab run (GPU)

This notebook trains and evaluates Sentinel on the **real-data track** only. It never mixes in simulator data.

Before running:
1. Runtime → Change runtime type → **T4 GPU** (or better).
2. Upload the official **UCI SMS Spam Collection** zip (`smsspamcollection.zip`) when asked. Without licensed legitimate messages the run can only learn scam *type* and *lures*, not scam-vs-legit.
3. Nothing here claims real-world deployment performance: report only the numbers in `metrics_real.json`, on the named test files.

In [ ]:
!nvidia-smi
!git clone https://github.com/TanviSaraswat8/Chakravyuh.git
%cd Chakravyuh
!git log -1 --oneline

In [ ]:
!pip -q install unsloth trl datasets langid
!pip -q install -r backend/requirements.txt

## 1. Fetch the pinned real datasets and verify SHA-256

In [ ]:
!python scripts/fetch_real_datasets.py fetch imc25_smishing sp24_gateway_phishing india_spam_sms_junioralive
!python scripts/fetch_real_datasets.py verify imc25_smishing sp24_gateway_phishing india_spam_sms_junioralive

## 2. Legitimate messages (official UCI file)
Upload `smsspamcollection.zip` from https://archive.ics.uci.edu/dataset/228/sms+spam+collection. The first registration pins its SHA-256 in the manifest; if it equals the digest of the GitHub mirror we audited (`mirror_sha256`), the mirror was faithful.

In [ ]:
from google.colab import files
up = files.upload()
!mkdir -p /content/uci && unzip -o smsspamcollection.zip -d /content/uci
!python scripts/fetch_real_datasets.py register uci_sms_spam --from /content/uci --pin
import json; m=json.load(open('data/registry/uci_sms_spam.json')); print('official == audited mirror:', m['sha256']==m['mirror_sha256'])

## 3. Validate, preprocess, build leakage-safe splits and SFT files

In [ ]:
!python scripts/validate_real_dataset.py imc25_smishing sp24_gateway_phishing india_spam_sms_junioralive uci_sms_spam
!python scripts/prepare_real_splits.py
!python sentinel/build_sft_real.py --config sentinel/configs/sentinel_real_v1.json
!cat sentinel/sft_real/sentinel_real_v1/build_manifest.json | head -60

## 4. Fine-tune (QLoRA). ~30–60 min on a T4 for 20k examples.

In [ ]:
!python sentinel/finetune.py --base unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit --data sentinel/sft_real/sentinel_real_v1 --out sentinel/outputs/sentinel_real_v1 --max-seq 512 --epochs 1

## 5. Predict on every test file, then score (track = REAL)

In [ ]:
!python sentinel/predict_hf.py --model sentinel/outputs/sentinel_real_v1/lora --data sentinel/sft_real/sentinel_real_v1 --out sentinel/outputs/sentinel_real_v1/predictions
!python sentinel/evaluate_real.py --data sentinel/sft_real/sentinel_real_v1 --pred sentinel/outputs/sentinel_real_v1/predictions --out sentinel/outputs/sentinel_real_v1/metrics_real.json

## 6. Model manifest (data + adapter hashes, config, metrics) and download

In [ ]:
!python sentinel/write_model_manifest.py --run sentinel/outputs/sentinel_real_v1 --data sentinel/sft_real/sentinel_real_v1 --config sentinel/configs/sentinel_real_v1.json
!cd sentinel/outputs && zip -qr sentinel_real_v1.zip sentinel_real_v1 -x '*/checkpoint-*'
from google.colab import files; files.download('sentinel/outputs/sentinel_real_v1.zip')

## Optional: unseen-family run
Repeat sections 3–6 with `sentinel/configs/sentinel_real_holdout_delivery_v1.json` (a separate model that never sees delivery scams).